In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Problem Statement

In this competition, participants are required to build machine learning or AI based systems capable of solving complex multiple choice questions. Each question contains a prompt along with five possible answer options labeled A, B, C, D, and E. The objective is to predict the top three most likely correct answers in ranked order.

The challenge focuses on evaluating a model’s ability to understand context, reason across options, and rank answers effectively. Participants are encouraged to experiment with a variety of approaches including transformer architectures, retrieval based pipelines, fine tuned language models, ensemble strategies, and efficient inference techniques.

Submissions will be evaluated using Mean Average Precision at 3 (MAP@3), where models receive higher scores when the correct answer appears earlier in their ranked predictions.



# Key EDA Highlights

(Full analysis in EDA.ipynb)

- **Dataset size**: 2,000 training questions, 500 test questions. No missing values in either set.

- **Class distribution (train `answer`)**: B (24.5%), C (23.0%), A (18.5%), D (17.9%), E (16.2%) -
  slightly imbalanced, with B/C over-represented and E least frequent. Relevant when interpreting
  accuracy vs. a majority-class baseline.

- **Prompt length**: mean ~18 words (range 3-51). Option length: mean ~26 words across A-E
  (range 1-118), roughly similar across all five options - so option length alone isn't a strong
  giveaway (confirmed below).

- **Longest-option heuristic**: picking the longest option as the answer is correct only 34.95%
  of the time - close to random (20% baseline), ruling out a simple length-based shortcut.

- **Duplicate questions**: 242 duplicate prompts within train; all duplicates share the same
  correct answer (no label conflicts). 8 duplicate prompts also found within test.

- **Train/test overlap (critical finding)**: 267 of 492 unique test prompts (54%) are **exact
  duplicates of training prompts**, and 95.5% of those also have identical answer options.
  This means over half the test set was effectively seen during training, which explains the
  near-perfect MAP@3 scores (~0.98-0.99) across all four modeling approaches, including simple
  classical ML, the task is easier than it appears due to leakage, not because every model is
  reasoning well.

- **Entities / Vocabulary**: heavy recurrence of the same physics/science entities
  (Newton, Einstein, Lorentz, Doppler, etc.) across train and test

# Model 1: Transformer From Scratch

Before using any pretrained language model, I first built a transformer encoder classifier
trained entirely from scratch (custom tokenizer + vocabulary, no pretrained embeddings). 

Key design choices: 

A word-level vocabulary (min frequency = 2 to drop rare noise tokens),
128-token max length, a 12-head/4-layer transformer encoder (516-dim embeddings), and a low
learning rate (3e-4) with AdamW + cosine annealing. Each question is scored by concatenating
it with one option at a time (`question SEP option`), so the model outputs one logit per
option and picks the top-3 by score.

## 1.1 configuration

In [ ]:
import torch
from collections import Counter
import string
from torch.utils.data import Dataset
import torch.nn as nn
import math
import pandas as pd
import wandb
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



class Config:

    # Data
    MAX_LEN = 128       #Max token sequence len
    BATCH_SIZE = 16     #samples per batch
    TEST_SIZE = 0.1     # size of validation data

    # Vocabulary
    MIN_FREQ = 2        #Min word freq for a word to be included in the vocab

    # Transformer
    EMBEDDING_DIM = 516 #Dimension of token embeddings
    NUM_HEADS = 12       #Num of attention heads
    NUM_LAYERS = 4      #Num of transformer encoder blocks
    FF_DIM = 1024       #hidden dimension of feed forward network
    DROPOUT = 0.4       # dropout probability

    # Training
    EPOCHS = 50        
    LR = 3e-4         # learning rate
    WEIGHT_DECAY = 1e-4 #L2 regularization strength

    #device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # seed value for reproducability
    SEED = 42



## 1.2 Vocabulary & Tokenizer

In [ ]:
# A simple tokenizer which converts text to lowercase if parameter lowercase set to True.
# It converts all the text including punctuation symbols to tokens.
class Tokenizer:

    def __init__(self, lowercase=False):
        self.lowercase = lowercase
        self.punctuation = list(string.punctuation)

    def tokenize(self, text):

        if text is None:
            return []

        text = str(text)

        if self.lowercase:
            text = text.lower()

        # Add spaces around punctuation
        for symbol in self.punctuation:
            text = text.replace(symbol, f" {symbol} ")

        # Convert to list of tokens
        tokens = text.split()

        return tokens

    def encode(self, text, vocab):

        tokens = self.tokenize(text)

        return vocab.encode(tokens)


    def decode(self, ids, vocab):   

        return vocab.decode(ids)



class Vocabulary:

    def __init__(self, min_freq=1):

        # Only words occuring for min_freq time are added to the vocabulary
        self.min_freq = min_freq

        # PAD token to make all sequences of the same length
        # UNK token for unknown words not found in the vocabulary
        # SEP seperates the question from the option
        self.word2idx = {"PAD": 0, "UNK": 1, "SEP": 2}

        self.idx2word = {0: "PAD", 1: "UNK", 2: "SEP"}


    def build(self, texts, tokenizer):

        counter = Counter()

        for text in texts:

            tokens = tokenizer.tokenize(text)

            counter.update(tokens)

        index = len(self.word2idx)

        for word, freq in counter.items():

            if word not in self.word2idx and freq >= self.min_freq:

                self.word2idx[word] = index
                self.idx2word[index] = word

                index += 1

    def encode(self, tokens):

        # returns idx 1 if token not found in vocab
        return [self.word2idx.get(token, 1) for token in tokens]


    def decode(self, ids):

        # returns UNK for idx 1
        return [self.idx2word.get(idx, "UNK") for idx in ids]


    def __len__(self):

        return len(self.word2idx)





## 1.3 Dataset

PyTorch models cannot directly read pandas DataFrames. The Dataset class converts each row of the DataFrame into tensors that the model can process. It also performs preprocessing such as combining the question and answer option, tokenization, encoding, padding, and label conversion.

In [ ]:

class MCQDataset(Dataset):

    def __init__(self, dataframe, tokenizer, vocab, max_len=Config.MAX_LEN):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.vocab = vocab
        self.max_len = max_len

        # sets True for training data, and false for test data 
        self.has_labels = "answer" in dataframe.columns

        self.answer_map = {
            "A": 0,
            "B": 1,
            "C": 2,
            "D": 3,
            "E": 4
        }

    def __len__(self):
        return len(self.df)

    def pad_sequence(self, ids):

        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
        else:
            ids += [0] * (self.max_len - len(ids))

        return ids

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        question = row["prompt"]

        options = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encoded_choices = []

        for option in options:
            # special SEP token to separate question from answer 
            text = question + " SEP " + option

            ids = self.tokenizer.encode(text, self.vocab)

            ids = self.pad_sequence(ids)

            encoded_choices.append(ids)
        
        # input_ids shape: (5, max_len)
        input_ids = torch.tensor(encoded_choices, dtype=torch.long)

        sample = {"input_ids": input_ids}

        if self.has_labels:
            # Convert string target into numerical target ('C' converted to 2)
            sample["label"] = torch.tensor(self.answer_map[row["answer"]], dtype=torch.long)

        return sample

## 1.4 Model Implementation 

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_seq_length, dropout=0.1):
        super().__init__()
        
        #Word embeddings
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        #Positional embeddings
        self.positional_embedding = nn.Embedding(
            num_embeddings=max_seq_length,
            embedding_dim=embedding_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        
        batch_size, seq_len = input_ids.shape

        #Create positions
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        positions = positions.expand(batch_size, seq_len)

        token_embedding = self.token_embedding(input_ids)

        # Learned positional embeddings are directly added to the embeddings
        positional_embedding = self.positional_embedding(positions)

        embeddings = token_embedding + positional_embedding

        return self.dropout(embeddings)



# attention
    
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim//num_heads

        self.Wq = nn.Linear(embedding_dim, embedding_dim)
        self.Wk = nn.Linear(embedding_dim, embedding_dim)
        self.Wv = nn.Linear(embedding_dim, embedding_dim)

        self.out_projection = nn.Linear(embedding_dim, embedding_dim)

    
    def forward(self, x, return_attention=False):

        # x shape: (batch_size, seq_len, embedding_dim)
        batch_size, seq_len, _ = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        # Split into multiple heads and transpose to: (batch_size, num_heads, seq_len, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # scores shape: (batch_size, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Scale scores to prevent them from becoming too large
        scores_scaled = scores / math.sqrt(self.head_dim)

        weights = torch.softmax(scores_scaled, dim=-1)
        
        # output shape after matmul: (batch_size, num_heads, seq_len, head_dim)
        output = torch.matmul(weights, V)

        # Permute and flatten back to original embedding space: (batch_size, seq_len, embedding_dim)
        output = output.transpose(1, 2).contiguous()

        output = output.view(batch_size, seq_len, self.embedding_dim)
        output = self.out_projection(output)

        if return_attention:
            return output, weights
        else:
            return output 


# feed forward

class FeedForward(nn.Module):
    def __init__(self, embedding_dim, ff_dim, dropout=0.1):
        super().__init__()

        # Expands the embedding dimension to a higher dimensional space (ff_dim)
        # This allows the model to process the attention outputs and learn more complex features
        self.fc1 = nn.Linear(embedding_dim, ff_dim)

        # GELU (Gaussian Error Linear Unit)
        #performs better than ReLU by providing smoother gradients.
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

        # Projects the expanded dimensional space back down to the original embedding_dim
        self.fc2 = nn.Linear(ff_dim, embedding_dim)

    def forward(self, x):

        # Initial x shape: (batch_size, seq_len, embedding_dim)
        x = self.fc1(x) # Shape transitions to: (batch_size, seq_len, ff_dim)

        #Applies GELU activation
        x = self.gelu(x)
        
        #dropout to reduce overfitting
        x = self.dropout(x)
        
        x = self.fc2(x) # Shape returns to: (batch_size, seq_len, embedding_dim)
        
        return x


# encoder block

class EncoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(embedding_dim, num_heads)

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.ffn = FeedForward(embedding_dim, ff_dim, dropout)

        self.norm2 = nn.LayerNorm(embedding_dim)

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        
        attention_output = self.attention(x)
        
        # Residual connection 1 followed by Layer Normalization (Post-LN)
        x = self.norm1(x + self.dropout(attention_output))

        ffn_output = self.ffn(x)
        
        # Residual connection 2 followed by Layer Normalization
        x = self.norm2(x + self.dropout(ffn_output))

        return x 
    
    
# transformer encoder

class TransformerEncoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim, max_seq_length, num_heads, num_layers, ff_dim, dropout=0.1):
        super().__init__()

        # Combines raw token embeddings with positional encodings.
        # This gives the model the identity of the words AND their sequential order.
        self.embedding = EmbeddingLayer(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            max_seq_length=max_seq_length,
            dropout=dropout
        )

        # stacked sequence of encoder blocks. 
        # nn.ModuleList is used, so pytorch for properly registers the parameters and updates them during backpropagation.
        self.layers = nn.ModuleList(
            [
                EncoderBlock(embedding_dim=embedding_dim, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
                for _ in range(num_layers)
            ]
        )

    def forward(self, input_ids):

        #token to embeddings
        x = self.embedding(input_ids) # (batch_size * num_choices, seq_len, embedding_dim)

        #sequentially pass the output of one encoder layer as the input to the next
        for layer in self.layers:
            x = layer(x)

        #shape of x: (batch_size * num_choices, seq_len, embedding_dim)
        return x



# classifier.py


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_seq_length, num_heads, num_layers, ff_dim, dropout=0.1):
        super().__init__()

        self.encoder = TransformerEncoder(
            vocab_size=vocab_size, 
            embedding_dim=embedding_dim,
            max_seq_length=max_seq_length,
            num_heads=num_heads,
            num_layers=num_layers,
            ff_dim=ff_dim,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(embedding_dim, 1)


    def forward(self, input_ids):

        # input_ids shape: (batch_size, num_choices=5, seq_len)
        batch_size, num_choices, seq_len = input_ids.shape

        # Flatten the batch and choice dimensions together so the encoder treats each multiple-choice option as an independent sequence.
        # New shape: (batch_size * 5, seq_len)
        input_ids = input_ids.reshape(batch_size * num_choices, seq_len)
        
        #encode
        x = self.encoder(input_ids) #(batch_size * 5, seq_len, embedding_dim)

        #mean pooling
        x = x.mean(dim=1) #(batch_size * 5, embedding_dim)

        x = self.dropout(x)

        # Project down to a single raw score (logit) per choice
        logits = self.classifier(x) #(batch_size * 5, 1)

        # back to shape (batch, 5)
        logits = logits.view(batch_size, num_choices)

        return logits



## 1.5 Metrics

In [ ]:
def accuracy(logits, labels):

    predictions = torch.argmax(logits, dim=1)

    correct = (predictions == labels).sum().item()

    return correct / len(labels)


def map_at_3(logits, labels):
    """
    Computes MAP@3 (Mean Average Precision) for multiple-choice predictions.
    If correct answer at first place in the prediction then awarded 1
    If at second position then 0.5
    If at third position then 0.33
    else 0 score is awarded
    """

    # Get the indices of the three highest-scoring predictions.
    top3 = torch.topk(logits, k=3, dim=1).indices

    score = 0

    for prediction, target in zip(top3, labels):

        prediction = prediction.tolist()
        target = target.item()

        if target == prediction[0]:
            score += 1

        elif target == prediction[1]:
            score += 0.5

        elif target == prediction[2]:
            score += 1 / 3

    return score / len(labels)


## 1.6 Training

In [ ]:
def train():

    device = Config.DEVICE

    train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

    train_df, val_df = train_test_split(train_df, test_size=Config.TEST_SIZE, random_state=Config.SEED, stratify=train_df["answer"])

    # Tokenizer & Vocabulary
    tokenizer = Tokenizer()
    vocab = Vocabulary(min_freq=Config.MIN_FREQ)

    texts = []

    texts.extend(train_df["prompt"].tolist())

    for col in ["A", "B", "C", "D", "E"]:
        texts.extend(train_df[col].tolist())

    vocab.build(texts, tokenizer)

    print(f"Vocabulary Size : {len(vocab)}")

    # Dataset
    train_dataset = MCQDataset(train_df, tokenizer, vocab, Config.MAX_LEN)

    val_dataset = MCQDataset(val_df, tokenizer, vocab, Config.MAX_LEN)

    # DataLoader

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

    # Model
    model = TransformerClassifier(
        vocab_size=len(vocab),
        embedding_dim=Config.EMBEDDING_DIM,
        max_seq_length=Config.MAX_LEN,
        num_heads=Config.NUM_HEADS,
        num_layers=Config.NUM_LAYERS,
        ff_dim=Config.FF_DIM,
        dropout=Config.DROPOUT
    ).to(device)

    # Loss
    # Cross-entropy compares predicted logits with the correct answer index.
    criterion = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY
    )

    #Scheduler
    #Gradually decreases the LR following a cosine curve
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=Config.EPOCHS
    )

    # Training Loop
    best_val_loss = float("inf")

    # wandb.init(
    #     entity="23f2001083-dl-genai-project",
    #     project="23f2001083-t22026",
    #     config={
    #         "embedding_dim": Config.EMBEDDING_DIM,
    #         "num_heads": Config.NUM_HEADS,
    #         "num_layers": Config.NUM_LAYERS,
    #         "ff_dim": Config.FF_DIM,
    #         "batch_size": Config.BATCH_SIZE,
    #         "learning_rate": Config.LR,
    #         "epochs": Config.EPOCHS,
    #         "max_len": Config.MAX_LEN,
    #         "dropout": Config.DROPOUT,
    #     }
    # )
    # wandb.watch(
    #     model,
    #     criterion,
    #     log="all",
    #     log_freq=100
    # )

    for epoch in range(Config.EPOCHS):

        model.train()

        train_loss = 0

        for batch in train_loader:

            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(input_ids)

            loss = criterion(logits, labels)

            loss.backward()

            #gradient clipping to prevent gradient exploding and improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)


        # Validation
        model.eval()

        val_loss = 0
        val_acc = 0
        val_map3 = 0

        # Disable gradient calculation to save memory and compute during validation
        with torch.no_grad():

            for batch in val_loader:

                input_ids = batch["input_ids"].to(device)
                labels = batch["label"].to(device)

                logits = model(input_ids)

                loss = criterion(logits, labels)

                val_loss += loss.item()
                val_acc += accuracy(logits, labels)
                val_map3 += map_at_3(logits, labels)

        val_loss /= len(val_loader)
        val_acc /= len(val_loader)
        val_map3 /= len(val_loader)

        print(
            f"Epoch {epoch+1}/{Config.EPOCHS}"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Acc {val_acc:.4f} "
            f" | MAP@3 {val_map3:.4f}"

        )
        # wandb.log({
        #     "train_loss": train_loss,
        #     "val_loss": val_loss,
        #     "val_accuracy": val_acc,
        #     "val_map3": val_map3,
        #     "epoch": epoch + 1,
        #     "learning_rate": optimizer.param_groups[0]["lr"]

        # })   

        scheduler.step()     

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            checkpoint = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "vocab": vocab,
                "config": Config
            }

            torch.save(checkpoint, "best_model.pth")
    print("Training Complete!")
    # wandb.finish()


train()


## 1.7 Inference

In [ ]:
OPTION_MAP = ["A", "B", "C", "D", "E"]


def predict():

    device = Config.DEVICE

    test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

    # Load the trained model
    checkpoint = torch.load(
        "best_model.pth",
        map_location=device,
        weights_only=False
    )

    #use the same vocabulary generated while training
    vocab = checkpoint["vocab"]

    tokenizer = Tokenizer()

    # Build model
    model = TransformerClassifier(
        vocab_size=len(vocab),
        embedding_dim=Config.EMBEDDING_DIM,
        max_seq_length=Config.MAX_LEN,
        num_heads=Config.NUM_HEADS,
        num_layers=Config.NUM_LAYERS,
        ff_dim=Config.FF_DIM,
        dropout=Config.DROPOUT
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    dataset = MCQDataset(
        test_df,
        tokenizer,
        vocab,
        Config.MAX_LEN
    )

    predictions = []

    #disable gradient computation for inference
    with torch.no_grad():

        for sample in dataset:

            #unsqueeze(0) converts shape from (num_choice(ie.5), SEQ_LEN) to (1, 5, SEQ_LEN)
            input_ids = sample["input_ids"].unsqueeze(0).to(device)

            logits = model(input_ids)

            #get the indices for top k=3 highest scoring predictions
            top3 = torch.topk(logits, k=3, dim=1).indices.squeeze(0)


            #Convert the numerical predicitons back to letter mappings
            prediction = " ".join(OPTION_MAP[idx] for idx in top3.tolist())
            predictions.append(prediction)

    # Kaggle submission
    submission = pd.DataFrame({
        "ID": test_df["id"],
        "Prediction": predictions
    })

    submission.to_csv("submission.csv", index=False)

    print("Submission saved to submission.csv")

predict()


# Model 2: Finetuned SciBERT



The question domain leans heavily scientific/technical (physics, astrophysics, named
scientists). SciBERT is pretrained on scientific text, so we finetune it
here to test whether domain-matched pretraining gives an edge over a general-purpose model.

## 2.1 Imports and configuration

In [ ]:
!pip install -q transformers datasets

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer
from transformers import AutoModel

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm.auto import tqdm


MODEL_NAME = "allenai/scibert_scivocab_uncased"

MAX_LEN = 256
BATCH_SIZE = 4
LR = 2e-5
EPOCHS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(DEVICE)

## 2.2 Data Loading and train-val split

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

df["label"] = df["answer"].map(label2id)

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(len(train_df))
print(len(valid_df))

## 2.3 Dataset class and Dataloader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class MCQDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        prompt = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encodings = []

        for choice in choices:

            encoding = tokenizer(
                prompt,
                choice,
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            encodings.append(encoding)

        input_ids = torch.stack(
            [e["input_ids"].squeeze(0) for e in encodings]
        )

        attention_mask = torch.stack(
            [e["attention_mask"].squeeze(0) for e in encodings]
        )

        label = torch.tensor(row["label"])

        return input_ids, attention_mask, label

In [ ]:
train_loader = DataLoader(
    MCQDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    MCQDataset(valid_df),
    batch_size=BATCH_SIZE,
    shuffle=False
)

## 2.4 SciBert model class

In [ ]:
class SciBertMCQ(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32
        )
        
        hidden = self.encoder.config.hidden_size

        self.classifier = nn.Linear(hidden,1)

    def forward(self,input_ids,attention_mask):

        batch_size = input_ids.shape[0]

        input_ids = input_ids.view(-1,MAX_LEN)
        attention_mask = attention_mask.view(-1,MAX_LEN)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        mask = attention_mask.unsqueeze(-1)
        
        cls = (outputs.last_hidden_state * mask).sum(dim=1)
        cls = cls / mask.sum(dim=1)
        
        logits = self.classifier(cls)

        logits = logits.view(batch_size,5)

        return logits

In [ ]:
model = SciBertMCQ().to(DEVICE)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

In [ ]:
# import wandb
# wandb.login()
# wandb.init(
#         entity="23f2001083-dl-genai-project",
#         project="23f2001083-t22026",
#         config={
#             "model": MODEL_NAME,
#             "epochs": EPOCHS,
#             "batch_size": BATCH_SIZE,
#             "lr": LR,
#             "max_len": MAX_LEN
#         }
            
#     )


## 2.5 Metrics 

In [ ]:
def accuracy(logits, labels):
    """
    Returns the number of correct predictions in the batch.
    """

    predictions = torch.argmax(logits, dim=1)

    correct = (predictions == labels).sum().item()

    return correct


def map_at_3(logits, labels):
    """
    Returns the cumulative MAP@3 score for the batch.
    """

    top3 = torch.topk(logits, k=3, dim=1).indices

    score = 0.0

    for prediction, target in zip(top3, labels):

        prediction = prediction.tolist()
        target = target.item()

        if prediction[0] == target:
            score += 1.0

        elif prediction[1] == target:
            score += 0.5

        elif prediction[2] == target:
            score += 1.0 / 3.0

    return score

## 2.6 Training

In [ ]:
def train_epoch():

    model.train()

    total_loss = 0

    for input_ids, attention_mask, labels in tqdm(train_loader):

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(
            input_ids,
            attention_mask
        )

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
def validate():

    model.eval()

    total_loss = 0

    total_correct = 0

    total_map3 = 0.0

    total_samples = 0

    with torch.no_grad():

        for input_ids, attention_mask, labels in tqdm(valid_loader):

            input_ids = input_ids.to(DEVICE)

            attention_mask = attention_mask.to(DEVICE)

            labels = labels.to(DEVICE)

            logits = model(
                input_ids,
                attention_mask
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += loss.item()

            total_correct += accuracy(logits, labels)

            total_map3 += map_at_3(logits, labels)

            total_samples += labels.size(0)

    avg_loss = total_loss / len(valid_loader)

    avg_acc = total_correct / total_samples

    avg_map3 = total_map3 / total_samples

    return avg_loss, avg_acc, avg_map3

In [ ]:
best_map3 = 0

for epoch in range(EPOCHS):

    train_loss = train_epoch()

    val_loss, val_acc, val_map3 = validate()

    if val_map3 > best_map3:

        best_map3 = val_map3

        torch.save(model.state_dict(), "best_scibert.pt")

    # wandb.log({
    #     "epoch": epoch + 1,
    #     "train_loss": train_loss,
    #     "valid_loss": val_loss,
    #     "valid_accuracy": val_acc,
    #     "valid_map3": val_map3
    # })

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Valid Loss : {val_loss:.4f}")
    print(f"Accuracy   : {val_acc:.4f}")
    print(f"MAP@3      : {val_map3:.4f}")

# wandb.finish()

### Result
Validation accuracy reaches 1.0 and MAP@3 = 0.994 by epoch 3. This is a very fast, very high
convergence for only ~1,800 training examples, consistent with the EDA finding that over
half the test set overlaps with train, and that the dataset has repeated phrasing.

In [ ]:
torch.save(model.state_dict(),"scibert_mcq.pt")

## 2.7 Inference

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        prompt = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encodings = []

        for choice in choices:

            encoding = tokenizer(
                prompt,
                choice,
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            encodings.append(encoding)

        input_ids = torch.stack(
            [e["input_ids"].squeeze(0) for e in encodings]
        )

        attention_mask = torch.stack(
            [e["attention_mask"].squeeze(0) for e in encodings]
        )

        return input_ids, attention_mask

In [ ]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test_loader = DataLoader(
    TestDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
model.eval()

all_predictions = []

letters = np.array(["A","B","C","D","E"])

with torch.no_grad():

    for input_ids, attention_mask in tqdm(test_loader):

        input_ids = input_ids.to(DEVICE)

        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)

        top3 = torch.topk(logits, k=3, dim=1).indices

        top3 = top3.cpu().numpy()

        for row in top3:

            prediction = " ".join(letters[row])

            all_predictions.append(prediction)

In [ ]:
submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": all_predictions
})

submission.to_csv("submission.csv", index=False)
print("Submission file created!")

# Model 3: Finetuned DeBERTa-v3-base


DeBERTa-v3 is a stronger general-purpose encoder than SciBERT (disentangled attention,
larger pretraining corpus). We test it here to see if a more powerful general model matches
or exceeds a smaller domain-specific one (SciBERT), independent of scientific pretraining.

## 3.1 Imports and Configuration

In [ ]:
!pip install -q transformers sentencepiece

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from tqdm import tqdm
from sklearn.model_selection import train_test_split


MODEL_NAME = "microsoft/deberta-v3-base"

MAX_LEN = 256
BATCH_SIZE = 4
LR = 1e-5
EPOCHS = 6
DROPOUT = 0.3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

## 3.2 Data Loading and Train-val split

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

df["label"] = df["answer"].map(label2id)

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(len(train_df))
print(len(valid_df))

## 3.3 Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class MCQDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row["prompt"]
        choices = [row["A"], row["B"], row["C"], row["D"], row["E"]]

        encodings = []
        for choice in choices:
            encoding = tokenizer(
                prompt,
                choice,
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt"
            )
            encodings.append(encoding)

        input_ids = torch.stack([e["input_ids"].squeeze(0) for e in encodings])
        attention_mask = torch.stack([e["attention_mask"].squeeze(0) for e in encodings])

        label = torch.tensor(row["label"])

        return input_ids, attention_mask, label

## 3.4 DeBerta Model

In [ ]:
class DebertaMCQ(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME,torch_dtype=torch.float32)
        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        batch_size = input_ids.shape[0]

        input_ids = input_ids.view(-1, MAX_LEN)
        attention_mask = attention_mask.view(-1, MAX_LEN)

        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1)
        pooled = (outputs.last_hidden_state * mask).sum(dim=1)
        pooled = pooled / mask.sum(dim=1)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        logits = logits.view(batch_size, 5)
        
        return logits


## 3.5 Metrics

In [ ]:
def accuracy(logits, labels):
    """
    Returns the number of correct predictions in the batch.
    """

    predictions = torch.argmax(logits, dim=1)

    correct = (predictions == labels).sum().item()

    return correct


def map_at_3(logits, labels):
    """
    Returns the cumulative MAP@3 score for the batch.
    """

    top3 = torch.topk(logits, k=3, dim=1).indices

    score = 0.0

    for prediction, target in zip(top3, labels):

        prediction = prediction.tolist()
        target = target.item()

        if prediction[0] == target:
            score += 1.0

        elif prediction[1] == target:
            score += 0.5

        elif prediction[2] == target:
            score += 1.0 / 3.0

    return score

## 3.6 Training

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0

    for input_ids, attention_mask, labels in tqdm(loader):
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_map3 = 0.0
    total_samples = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(loader):
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_correct += accuracy(logits, labels)
            total_map3 += map_at_3(logits, labels)
            total_samples += labels.size(0)

    avg_loss = total_loss / len(loader)
    avg_acc = total_correct / total_samples
    avg_map3 = total_map3 / total_samples

    return avg_loss, avg_acc, avg_map3

In [ ]:
!pip install wandb -q
# import wandb
# wandb.login()
# wandb.init(
#         entity="23f2001083-dl-genai-project",
#         project="23f2001083-t22026",
#         config={
#             "model": MODEL_NAME,
#             "epochs": EPOCHS,
#             "batch_size": BATCH_SIZE,
#             "lr": LR,
#             "max_len": MAX_LEN,
#             "dropout": DROPOUT
#         }
            
#     )

In [ ]:
# DataLoaders
train_loader = DataLoader(
    MCQDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    MCQDataset(valid_df),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Model
model = DebertaMCQ().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

# Scheduler
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Training Loop
best_map3 = 0.0
best_epoch = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion
    )

    val_loss, val_acc, val_map3 = validate(
        model,
        valid_loader,
        criterion
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {val_loss:.4f} | "
        f"Accuracy: {val_acc:.4f} | "
        f"MAP@3: {val_map3:.4f}"
    )

    # wandb.log({
    #     "epoch": epoch + 1,
    #     "train_loss": train_loss,
    #     "valid_loss": val_loss,
    #     "valid_accuracy": val_acc,
    #     "valid_map3": val_map3
    # })


    if val_map3 > best_map3:
        best_map3 = val_map3
        best_epoch = epoch + 1

        torch.save(model.state_dict(), "best_model.pt")

        print(f" Best model saved (Epoch {best_epoch})")

print("Training Complete!")
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation MAP@3: {best_map3:.4f}")
print("Model saved as: best_model.pt")

# wandb.finish()

## 3.7 Inference

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row["prompt"]
        choices = [row["A"], row["B"], row["C"], row["D"], row["E"]]

        encodings = []
        for choice in choices:
            encoding = tokenizer(
                prompt,
                choice,
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt"
            )
            encodings.append(encoding)

        input_ids = torch.stack([e["input_ids"].squeeze(0) for e in encodings])
        attention_mask = torch.stack([e["attention_mask"].squeeze(0) for e in encodings])

        return input_ids, attention_mask

In [ ]:
# Load Test Data
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test_loader = DataLoader(
    TestDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Load Best Model
model = DebertaMCQ().to(DEVICE)
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))

model.eval()

# Predict
all_probs = []

with torch.no_grad():
    for input_ids, attention_mask in tqdm(test_loader):

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())

# Shape: (num_test_samples, num_classes)
avg_probs = np.concatenate(all_probs, axis=0)

print("Prediction shape:", avg_probs.shape)

In [ ]:
letters = np.array(["A", "B", "C", "D", "E"])

all_predictions = []
for row in avg_probs:
    top3_idx = np.argsort(row)[::-1][:3]
    prediction = " ".join(letters[top3_idx])
    all_predictions.append(prediction)

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": all_predictions
})
submission.to_csv("submission.csv", index=False)
submission.head()

# Model4: Classical ML approach

To test whether transformer-scale models are even necessary for this task, I built a
classical pipeline: TF-IDF features over (question, option) pairs, framed as binary
"is this option correct?" classification, scored with Logistic Regression, SGD, Naive Bayes,
and CatBoost.

## 4.1 Imports and Config

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

SEED = 42


## 4.2 Data Loading

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")


train_df, valid_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["answer"],
    random_state=SEED
)

In [ ]:
OPTIONS = ["A", "B", "C", "D", "E"]

# Function to create question option pair
def build_pairs(df, has_label=True):

    rows = []

    for qid, row in df.reset_index(drop=True).iterrows():

        for option in OPTIONS:

            rows.append({
                "qid": qid,
                "option": option,
                "pair_text": f"Question: {row['prompt']} Answer Choice: {row[option]}",
                "label": int(row["answer"] == option) if has_label else None
            })

    return pd.DataFrame(rows)

train_pairs = build_pairs(train_df, has_label=True)
valid_pairs = build_pairs(valid_df, has_label=True)

y_train = train_pairs["label"].values
y_valid = valid_pairs["label"].values

## 4.3 TfIdf Vectorization

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            max_features=120000,
            ngram_range=(1, 3),
            min_df=2,
            sublinear_tf=True,
        )
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            max_features=90000,
            min_df=2,
            sublinear_tf=True,
        )
    )
])

X_train = tfidf_features.fit_transform(train_pairs["pair_text"])
X_valid = tfidf_features.transform(valid_pairs["pair_text"])

## 4.4 Classical Models

In [ ]:
from sklearn.pipeline import Pipeline

cat_pipe = Pipeline([
    ("features", tfidf_features),
    ("clf", CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=SEED
    ))
])
print("Fitting Model")
cat_pipe.fit(train_pairs["pair_text"], y_train)
print("Completed!")

## 4.5 Metrics

In [ ]:
OPTIONS = ["A", "B", "C", "D", "E"]

def map3_score(score_matrix, answers):

    score = 0

    for scores, gt in zip(score_matrix, answers):

        order = np.argsort(scores)[::-1]

        preds = [OPTIONS[i] for i in order[:3]]

        if preds[0] == gt:
            score += 1
        elif preds[1] == gt:
            score += 0.5
        elif preds[2] == gt:
            score += 1/3

    return score / len(answers)

In [ ]:
valid_scores = cat_pipe.predict_proba(valid_pairs["pair_text"])[:, 1]
valid_matrix = valid_scores.reshape(len(valid_df), 5)

pred_idx = valid_matrix.argmax(axis=1)
pred_answer = [OPTIONS[i] for i in pred_idx]

acc = accuracy_score(
    valid_df["answer"],
    pred_answer
)

map3 = map3_score(
    valid_matrix,
    valid_df["answer"].values
)

print(f"CatBoost Accuracy : {acc}")
print(f"CatBoost MAP@3    : {map3}")

## 4.7 Inference

In [ ]:
test_pairs = build_pairs(
    test_df,
    has_label=False
)

test_scores = cat_pipe.predict_proba(test_pairs["pair_text"])[:, 1]

test_matrix = test_scores.reshape(len(test_df), 5)

In [ ]:
predictions = []

for scores in test_matrix:

    order = np.argsort(scores)[::-1]

    top3 = [OPTIONS[i] for i in order[:3]]

    predictions.append(" ".join(top3))

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()